In [0]:
#PARCIAL 2

#Realizado por : JUAN JACOBO OBANDO FRANCO - DIEGO FERNANDO SERNA

#PREPARACIÓN DE LOS DATOS

import pandas as pd

# Cargar las tablas y convertirlas a Pandas
ft = spark.read.format("delta").table("default.electiva_FT").toPandas()
desempleados = spark.read.format("delta").table("default.electiva_desempleados").toPandas()
cgse = spark.read.format("delta").table("default.electiva_CGSE").toPandas()
ocupados = spark.read.format("delta").table("default.electiva_ocupados").toPandas()

# Agregar sexo y edad a los registros de desempleo
llaves = ["DIRECTORIO", "SECUENCIA_P", "ORDEN"]
caracteristicas = cgse[llaves + ["P3271", "P6040"]]
desempleados = desempleados.merge(caracteristicas, on=llaves, how="left")

# Códigos DANE y nombres de departamentos
dpto_nombre = {
    5: "ANTIOQUIA", 8: "ATLANTICO", 11: "BOGOTA", 13: "BOLIVAR",
    15: "BOYACA", 17: "CALDAS", 18: "CAQUETA", 19: "CAUCA",
    20: "CESAR", 23: "CORDOBA", 25: "CUNDINAMARCA", 27: "CHOCO",
    41: "HUILA", 44: "GUAJIRA", 47: "MAGDALENA", 50: "META",
    52: "NARIÑO", 54: "NORTE DE SANTANDER", 63: "QUINDIO",
    66: "RISARALDA", 68: "SANTANDER", 70: "SUCRE", 73: "TOLIMA",
    76: "VALLE DEL CAUCA", 81: "ARAUCA", 85: "CASANARE",
    86: "PUTUMAYO", 88: "SAN ANDRES", 91: "AMAZONAS",
    94: "GUAINIA", 95: "GUAVIARE", 97: "VAUPES", 99: "VICHADA"
}

# Función para clasificar edades
def grupo_edad(edad):
    if 18 <= edad < 24: return "18 a 24"
    if 24 <= edad < 30: return "24 a 30"
    if 30 <= edad < 40: return "30 a 40"
    if 40 <= edad < 50: return "40 a 50"
    if 50 <= edad < 60: return "50 a 60"
    if edad >= 60: return "60 en adelante"
    return None

# Función para calcular promedios ponderados de ingreso
def preparar_ingresos(df):
    df = df.copy()
    df["INGLABO"] = pd.to_numeric(df["INGLABO"], errors="coerce")
    df["FEX_C18"] = pd.to_numeric(df["FEX_C18"], errors="coerce")
    return df[(df["INGLABO"] > 0) & (df["FEX_C18"] > 0)].copy()


#PUNTO 1 — INDICADORES NACIONALES DEL MERCADO LABORAL


pet = ft.loc[ft["PET"] == 1, "FEX_C18"].sum()
poblacion_total = cgse["FEX_C18"].sum()
no_pet = poblacion_total - pet

ocupados_total = ocupados.loc[ocupados["OCI"] == 1, "FEX_C18"].sum()
desocupados_total = desempleados.loc[desempleados["DSI"] == 1, "FEX_C18"].sum()
pea = ocupados_total + desocupados_total
pei = pet - pea

tasa_ocupacion = ocupados_total / pet * 100
tasa_desempleo = desocupados_total / pea * 100

print(f"PET: {pet:,.0f}")
print(f"Fuera de la edad de trabajar: {no_pet:,.0f}")
print(f"PEA: {pea:,.0f}")
print(f"PEI: {pei:,.0f}")
print(f"Tasa de ocupación: {tasa_ocupacion:.2f}%")
print(f"Tasa de desempleo: {tasa_desempleo:.2f}%")


#Respuesta reportada:
# PET: 41.202.631
# Fuera de la edad de trabajar: 11.346.181
#PEA: 26.593.429
#EI: 14.609.202
#Tasa de ocupación: 59,36%
#Tasa de desempleo: 8,03%

#Interpretación: De los cerca de 52,5 millones de habitantes que cubre la encuesta, 41,2 millones (78%) tienen edad para trabajar y 11,3 millones son menores de 15 años. De la PET, 26,6 millones participan en el mercado laboral y 14,6 millones están inactivos, por ejemplo, por estudiar, realizar labores del hogar o estar jubilados. De la PEA, el 8,03% está desempleado, es decir, cerca de 2,1 millones de personas. La tasa de ocupación de 59,36% indica que casi seis de cada diez personas en edad de trabajar tienen empleo.

#PUNTO 2 — DESEMPLEO POR DEPARTAMENTO Y SEXO

# Agregar sexo a los ocupados
oc = ocupados[ocupados["OCI"] == 1].merge(
    cgse[llaves + ["P3271"]], on=llaves, how="inner"
)
des = desempleados[desempleados["DSI"] == 1]

# Sumar ocupados y desempleados por departamento y sexo
oc_sum = oc.groupby(["DPTO", "P3271"])["FEX_C18"].sum().rename("Ocupados")
des_sum = des.groupby(["DPTO", "P3271"])["FEX_C18"].sum().rename("Desocupados")
tabla = pd.concat([oc_sum, des_sum], axis=1).fillna(0).reset_index()
tabla["PEA"] = tabla["Ocupados"] + tabla["Desocupados"]
tabla["Tasa"] = tabla["Desocupados"] / tabla["PEA"] * 100
tabla["Sexo"] = tabla["P3271"].map({1: "Hombre", 2: "Mujer"})
tabla["Departamento"] = tabla["DPTO"].map(dpto_nombre)

resultado_p2 = tabla.pivot(index="Departamento", columns="Sexo", values="Tasa").reset_index()

# Tasa total por departamento
total = tabla.groupby("Departamento")[["Desocupados", "PEA"]].sum()
total["TD_Total"] = total["Desocupados"] / total["PEA"] * 100
display(resultado_p2)
resumen_p2 = resultado_p2.merge(total[["TD_Total"]].reset_index(), on="Departamento")
resumen_p2 = resumen_p2.rename(columns={
    "Hombre": "TD_Hombres (%)", "Mujer": "TD_Mujeres (%)",
    "TD_Total": "TD_Total (%)"
})
display(resumen_p2)

print("Departamento con mayor desempleo:")
print(total.sort_values("TD_Total", ascending=False).head(3)[["TD_Total"]])
print("Departamento con menor desempleo:")
print(total.sort_values("TD_Total").head(3)[["TD_Total"]])


#Respuesta :
#General: Putumayo (22,18%); menor: Vaupés (2,46%).
#Hombres: mayor en San Andrés (20,98%); menor en Vaupés (3,00%).
#Mujeres: mayor en Putumayo (25,02%); menor en Vaupés (1,78%).

#Interpretación: Putumayo tiene la tasa total más alta (22,18%), casi tres veces el promedio nacional (8,03%); entre las mujeres llega al 25,02%. Vaupés registra la tasa más baja en general y para ambos sexos, pero su muestra puede ser pequeña. En la mayoría de los departamentos, el desempleo femenino supera al masculino. La brecha más marcada aparece en Bolívar (17,4% mujeres frente a 6,4% hombres) y Boyacá (14,1% frente a 7,8%); Bogotá y Amazonas son las excepciones mencionadas en el trabajo original.

#PUNTO 3 — DESEMPLEO POR DEPARTAMENTO Y EDAD


cgse["Grupo_Edad"] = cgse["P6040"].apply(grupo_edad)
caracteristicas_edad = cgse[llaves + ["Grupo_Edad"]]

oc = ocupados[ocupados["OCI"] == 1].merge(caracteristicas_edad, on=llaves, how="inner")
des = desempleados[desempleados["DSI"] == 1].merge(caracteristicas_edad, on=llaves, how="inner")
oc = oc[oc["Grupo_Edad"].notna()]
des = des[des["Grupo_Edad"].notna()]

oc_sum = oc.groupby(["DPTO", "Grupo_Edad"])["FEX_C18"].sum().rename("Ocupados")
des_sum = des.groupby(["DPTO", "Grupo_Edad"])["FEX_C18"].sum().rename("Desocupados")
resultado = pd.concat([oc_sum, des_sum], axis=1).fillna(0).reset_index()
resultado["PEA"] = resultado["Ocupados"] + resultado["Desocupados"]
resultado["Tasa"] = resultado["Desocupados"] / resultado["PEA"] * 100
resultado["Departamento"] = resultado["DPTO"].map(dpto_nombre)

mayor = resultado.loc[resultado.groupby("Departamento")["Tasa"].idxmax(),
                      ["Departamento", "Grupo_Edad", "Tasa"]].rename(
    columns={"Grupo_Edad": "Edad Mayor TD", "Tasa": "Mayor TD (%)"})
menor = resultado.loc[resultado.groupby("Departamento")["Tasa"].idxmin(),
                      ["Departamento", "Grupo_Edad", "Tasa"]].rename(
    columns={"Grupo_Edad": "Edad Menor TD", "Tasa": "Menor TD (%)"})
resultado_p3 = mayor.merge(menor, on="Departamento")
display(resultado_p3)


#Respuesta: La tabla muestra el grupo con mayor y menor desempleo en cada departamento.

#Interpretación: Las tasas más altas se concentran sobre todo en jóvenes de 18 a 24 años (por ejemplo, Bolívar 30,9%, Casanare 29,2%, Córdoba 23,9% y Cundinamarca 23,1%) o de 24 a 30 años (Arauca 22,8%, Boyacá 22,6% y Chocó 19,2%). Las tasas más bajas suelen aparecer entre personas de 50 años o más, con valores entre 0% y 5,4%. Esto refleja las dificultades de los jóvenes para entrar al mercado laboral. En mayores de 60 años, algunas personas dejan de buscar empleo y ya no se cuentan como desempleadas.

# PUNTO 4 — TRES DEPARTAMENTOS CON MENOR DESEMPLEO EN MUJERES DE 18 A 25 AÑOS


oc = ocupados[ocupados["OCI"] == 1].merge(
    cgse[llaves + ["P3271", "P6040"]], on=llaves, how="inner"
)
des = desempleados[desempleados["DSI"] == 1]

filtro_oc = (oc["P3271"] == 2) & oc["P6040"].between(18, 25)
filtro_des = (des["P3271"] == 2) & des["P6040"].between(18, 25)
oc = oc[filtro_oc]
des = des[filtro_des]

oc_sum = oc.groupby("DPTO")["FEX_C18"].sum().rename("Ocupados")
des_sum = des.groupby("DPTO")["FEX_C18"].sum().rename("Desocupados")
resultado = pd.concat([oc_sum, des_sum], axis=1).fillna(0).reset_index()
resultado["PEA"] = resultado["Ocupados"] + resultado["Desocupados"]
resultado["Tasa"] = resultado["Desocupados"] / resultado["PEA"] * 100
resultado["Departamento"] = resultado["DPTO"].map(dpto_nombre)
top3_menor_td_mujeres = resultado[["Departamento", "Tasa"]].sort_values("Tasa").head(3)
display(top3_menor_td_mujeres)


#Respuesta reportada:
#Vaupés: 0,00%
#Nariño: 4,29%
#Amazonas: 4,87%

#Interpretación: Vaupés tiene una tasa reportada de 0%, posiblemente porque la muestra incluye pocas mujeres de 18 a 25 años y ninguna aparece como desempleada. Nariño (4,29%) y Amazonas (4,87%) también registran tasas bajas, por debajo del promedio nacional reportado (8,03%).

#PUNTO 5 — INGRESO LABORAL PROMEDIO EN CINCO DEPARTAMENTOS

deptos = [66, 17, 63, 5, 76]
ingresos = preparar_ingresos(ocupados[(ocupados["OCI"] == 1) & ocupados["DPTO"].isin(deptos)])
ingresos["Ingreso_Ponderado"] = ingresos["INGLABO"] * ingresos["FEX_C18"]

tabla = ingresos.groupby("DPTO")[["Ingreso_Ponderado", "FEX_C18"]].sum().reset_index()
tabla["Promedio INGLABO"] = tabla["Ingreso_Ponderado"] / tabla["FEX_C18"]
tabla["Departamento"] = tabla["DPTO"].map(dpto_nombre)
resultado_p5 = tabla[["Departamento", "Promedio INGLABO"]].sort_values(
    "Promedio INGLABO", ascending=False
).reset_index(drop=True)
display(resultado_p5)


#(promedio mensual):
# Antioquia: $2.486.207
 #Risaralda: $2.234.061
 #Caldas: $2.226.450
 #Quindío: $2.171.560
 #Valle del Cauca: $2.147.996

#Interpretación: los ingresos promedio de los cinco departamentos son similares (alrededor de $2,2 millones mensuales). Antioquia presenta el promedio más alto de los cinco departamentos, con $2.486.207.

#PUNTO 6 — INGRESO POR DEPARTAMENTO, SEXO Y EDAD

deptos = [66, 17, 63, 5, 76]
base = ocupados[(ocupados["OCI"] == 1) & ocupados["DPTO"].isin(deptos)].merge(
    cgse[llaves + ["P6040", "P3271"]], on=llaves, how="inner"
)
base["Grupo_Edad"] = base["P6040"].apply(grupo_edad)
base["Sexo"] = base["P3271"].map({1: "Hombre", 2: "Mujer"})
base = preparar_ingresos(base)
base = base[base["Grupo_Edad"].notna()].copy()
base["Ingreso_Ponderado"] = base["INGLABO"] * base["FEX_C18"]

tabla = base.groupby(["DPTO", "Sexo", "Grupo_Edad"])[
    ["Ingreso_Ponderado", "FEX_C18"]].sum().reset_index()
tabla["Promedio INGLABO"] = tabla["Ingreso_Ponderado"] / tabla["FEX_C18"]
tabla["Departamento"] = tabla["DPTO"].map(dpto_nombre)
resultado_p6 = tabla[["Departamento", "Sexo", "Grupo_Edad", "Promedio INGLABO"]].sort_values(
    ["Departamento", "Sexo", "Grupo_Edad"]).reset_index(drop=True)
display(resultado_p6)


#Interpretación: En casi todos los rangos de edad, el ingreso promedio de los hombres suele ser mayor que el de las mujeres, y la diferencia aumenta con la edad. En Antioquia, el texto original indica una brecha importante entre hombres y mujeres de 60 años o más y una diferencia menor entre los de 18 a 24 años; los montos de esa comparación aparecen incompletos en el original. Los ingresos tienden a aumentar hasta edades cercanas a los 40–60 años y luego bajar. Esto puede relacionarse con la experiencia laboral, el tipo de ocupación y las interrupciones asociadas al trabajo de cuidado.

#PUNTO 7 — OFICIOS CON MAYOR INGRESO PROMEDIO EN COLOMBIA


ingresos = preparar_ingresos(ocupados[ocupados["OCI"] == 1])
ingresos["Ingreso_Ponderado"] = ingresos["INGLABO"] * ingresos["FEX_C18"]

tabla = ingresos.groupby("OFICIO_C8")[["Ingreso_Ponderado", "FEX_C18"]].sum().reset_index()
tabla["Promedio INGLABO"] = tabla["Ingreso_Ponderado"] / tabla["FEX_C18"]
top_oficios = tabla.nlargest(10, "Promedio INGLABO").copy()

oficio_nombre = {
    1111: "Directores formuladores de políticas y normas",
    1120: "Directores y gerentes generales",
    2211: "Médicos generales",
    2212: "Médicos especialistas",
    2612: "Jueces",
    1223: "Directores de investigación y desarrollo",
    1330: "Directores de servicios de tecnología de información",
    2422: "Profesionales en políticas de administración",
    2522: "Administradores de sistemas",
    1323: "Directores de empresas de construcción",
    3321: "Agentes de seguros",
    2145: "Ingenieros químicos",
    2512: "Desarrolladores de software",
    9611: "Recolectores de basura y material reciclable",
    3139: "Técnicos en control de procesos"
}
top_oficios["Oficio"] = top_oficios["OFICIO_C8"].map(oficio_nombre).fillna(
    top_oficios["OFICIO_C8"].astype(str))
display(top_oficios[["Oficio", "OFICIO_C8", "FEX_C18", "Promedio INGLABO"]])


#Respuesta: El archivo original no incluye los diez resultados numéricos de esta tabla; se obtienen al ejecutar el código con las tablas de datos.

#PUNTO 8 — DIFERENCIAS DE INGRESO POR SEXO EN CUATRO DEPARTAMENTOS

deptos = [17, 63, 5, 76]
oficio_elegido = 2211

base = ocupados[(ocupados["OCI"] == 1) &
                ocupados["DPTO"].isin(deptos) &
                (ocupados["OFICIO_C8"] == oficio_elegido)].merge(
    cgse[llaves + ["P3271"]], on=llaves, how="inner")
base = preparar_ingresos(base)
base["Ingreso_Ponderado"] = base["INGLABO"] * base["FEX_C18"]
base["Sexo"] = base["P3271"].map({1: "Hombre", 2: "Mujer"})

tabla = base.groupby(["DPTO", "Sexo"])[["Ingreso_Ponderado", "FEX_C18"]].sum().reset_index()
tabla["Promedio INGLABO"] = tabla["Ingreso_Ponderado"] / tabla["FEX_C18"]
tabla["Departamento"] = tabla["DPTO"].map(dpto_nombre)
resultado_p8 = tabla.pivot(index="Departamento", columns="Sexo", values="Promedio INGLABO").reset_index()
print(f"Diferencias salariales por sexo — oficio CIUO-08 {oficio_elegido} (médicos generales)")
display(resultado_p8)


#PUNTO 9 — OFICIOS CON MAYOR Y MENOR INGRESO EN QUINDÍO, POR SEXO

base = ocupados[(ocupados["OCI"] == 1) & (ocupados["DPTO"] == 63)].merge(
    cgse[llaves + ["P3271"]], on=llaves, how="inner")
base = preparar_ingresos(base)
base["Ingreso_Ponderado"] = base["INGLABO"] * base["FEX_C18"]
base["Sexo"] = base["P3271"].map({1: "Hombre", 2: "Mujer"})

tabla = base.groupby(["Sexo", "OFICIO_C8"])[["Ingreso_Ponderado", "FEX_C18"]].sum().reset_index()
tabla["Promedio INGLABO"] = tabla["Ingreso_Ponderado"] / tabla["FEX_C18"]

extremos = []
for sexo in ["Hombre", "Mujer"]:
    datos = tabla[tabla["Sexo"] == sexo]
    extremos.append(datos.loc[datos["Promedio INGLABO"].idxmax()].to_dict())
    extremos.append(datos.loc[datos["Promedio INGLABO"].idxmin()].to_dict())

resumen_quindio = pd.DataFrame(extremos)
resumen_quindio["Categoría"] = ["Mayor Hombres", "Menor Hombres", "Mayor Mujeres", "Menor Mujeres"]
resumen_quindio["Oficio"] = resumen_quindio["OFICIO_C8"].map(oficio_nombre).fillna(
    resumen_quindio["OFICIO_C8"].astype(str))
resultado_p9 = resumen_quindio[["Categoría", "Sexo", "Oficio", "OFICIO_C8", "Promedio INGLABO"]]
display(resultado_p9)

for _, fila in resumen_quindio.iterrows():
    print(f"{fila['Categoría']}: {fila['Oficio']} (código {fila['OFICIO_C8']}) — promedio ${fila['Promedio INGLABO']:,.0f}")


#Respuesta:
